# Faithfulness

## Model Preparation

In [ ]:
import pandas as pd
import torch
from busi_xai.data import BUSIDataset, build_image_transform
from busi_xai.xai import explain_single_prediction
from busi_xai.config import SPLITS_DIR,MODELS_DIR
from busi_xai import model
from busi_xai.model import build_resnet50, load_checkpoint
import numpy as np

# Load the existing test split.
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

test_dataset = BUSIDataset(
    dataframe=test_df,
    image_column="processed_image",
    class_column="class_name",
    image_transform=build_image_transform(train=False),
    mask_column="processed_mask",
)

sample_index = 0
sample = test_dataset[sample_index]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_resnet50(num_classes=2)

checkpoint_path = MODELS_DIR / "resnet50_best.pt"

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

model = model.to(device)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Al Fares/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


  5%|▍         | 4.50M/97.8M [00:20<07:04, 230kB/s]


KeyboardInterrupt: 

## Calculate deletion faithfulness

In [10]:
import json
import numpy as np
import torch
from busi_xai.config import XAI_OUTPUT_DIR

from busi_xai.xai_evaluation import (
    calculate_deletion_faithfulness
)

# Existing experiment outputs
XAI_RECORDS_PATH = ( XAI_OUTPUT_DIR / "xai_records.json" )

GRADCAM_MAP_PATH = (
    XAI_OUTPUT_DIR / "gradcam/dataset_index_000.npy"
)

# Load the existing record
with open(XAI_RECORDS_PATH, "r") as f:
    records = json.load(f)

record = next(
    r for r in records
    if r["method"] == "gradcam"
    and r["dataset_index"] == 0
)

# Load the existing attribution map
attribution_map = np.load(GRADCAM_MAP_PATH)

print("Record:")
print(record)

print("\nAttribution map:")
print("Shape:", attribution_map.shape)
print("Dtype:", attribution_map.dtype)
print("Range:", float(attribution_map.min()), float(attribution_map.max()))

# Reuse the existing test dataset and model from the current notebook.
sample = test_dataset[0]

image = sample["image"].unsqueeze(0).to(device)

result = calculate_deletion_faithfulness(
    model=model,
    image_tensor=image,
    attribution_map=attribution_map,
    target_class=record["predicted_class"],
    device=device,
    percentile=90.0,
    baseline_value=0.0,
)

print("\n=== Single-image faithfulness validation ===")
for key, value in result.items():
    print(f"{key}: {value}")

Record:
{'image_id': 0, 'dataset_index': 0, 'true_label': 0, 'predicted_class': 0, 'predicted_probability': 0.990011990070343, 'correct': True, 'method': 'gradcam', 'runtime_seconds': 0.03325396800028102, 'localisation_percentile': 90.0, 'explanation_pixels': 5018, 'ground_truth_pixels': 472, 'intersection_pixels': 245, 'union_pixels': 5245, 'iou': 0.04671115347950429, 'dice': 0.08925318761384335}

Attribution map:
Shape: (224, 224)
Dtype: float32
Range: 0.0 1.0


FileNotFoundError: [Errno 2] No such file or directory: '\\content\\drive\\MyDrive\\BUSI-XAI\\data\\processed\\images\\benign\\benign (189).png'

## Evaluate faithfulness

In [ ]:
from pathlib import Path
import json
from busi_xai.config import XAI_OUTPUT_DIR
from busi_xai.xai_evaluation import (
    evaluate_faithfulness_records
)


XAI_RECORDS_PATH = XAI_OUTPUT_DIR / "xai_records.json"

with open(XAI_RECORDS_PATH, "r") as f:
    xai_records = json.load(f)

faithfulness_records = evaluate_faithfulness_records(
    model=model,
    test_dataset=test_dataset,
    records=xai_records,
    xai_output_dir=XAI_OUTPUT_DIR,
    device=device,
    percentile=90.0,
    baseline_value=0.0,
)

FAITHFULNESS_OUTPUT_PATH = (
    XAI_OUTPUT_DIR
    / "xai_faithfulness_records.json"
)

FAITHFULNESS_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(FAITHFULNESS_OUTPUT_PATH, "w") as f:
    json.dump(
        faithfulness_records,
        f,
        indent=2,
    )


print("Faithfulness records:", len(faithfulness_records))
print("First record:")
print(faithfulness_records[0])
print("Last record:")
print(faithfulness_records[-1])from pathlib import Path

print("Faithfulness records saved.")
print("Path:", FAITHFULNESS_OUTPUT_PATH)
print("Records:", len(faithfulness_records))
print("File exists:", FAITHFULNESS_OUTPUT_PATH.exists())


FileNotFoundError: [Errno 2] No such file or directory: '\\content\\drive\\MyDrive\\BUSI-XAI\\data\\processed\\images\\benign\\benign (189).png'

## Faithfulness aggregations

In [ ]:
import json
import statistics
from busi_xai.config import METRICS_DIR

with open(FAITHFULNESS_OUTPUT_PATH, "r") as f:
    saved_faithfulness_records = json.load(f)

faithfulness_aggregate = {}

for method in ["gradcam", "lime", "shap"]:
    method_records = [
        record
        for record in saved_faithfulness_records
        if record["method"] == method
    ]

    probability_drops = [
        float(record["probability_drop"])
        for record in method_records
    ]

    original_probabilities = [
        float(record["original_probability"])
        for record in method_records
    ]

    perturbed_probabilities = [
        float(record["perturbed_probability"])
        for record in method_records
    ]

    deleted_fractions = [
        float(record["deleted_fraction"])
        for record in method_records
    ]

    faithfulness_aggregate[method] = {
        "n": len(method_records),
        "original_probability_mean": statistics.mean(
            original_probabilities
        ),
        "perturbed_probability_mean": statistics.mean(
            perturbed_probabilities
        ),
        "probability_drop_mean": statistics.mean(
            probability_drops
        ),
        "probability_drop_median": statistics.median(
            probability_drops
        ),
        "probability_drop_std": statistics.stdev(
            probability_drops
        ),
        "probability_drop_min": min(
            probability_drops
        ),
        "probability_drop_max": max(
            probability_drops
        ),
        "deleted_fraction_mean": statistics.mean(
            deleted_fractions
        ),
    }

FAITHFULNESS_AGGREGATE_PATH = (
    METRICS_DIR
    / "xai_faithfulness_aggregate_metrics.json"
)

with open(FAITHFULNESS_AGGREGATE_PATH, "w") as f:
    json.dump(
        faithfulness_aggregate,
        f,
        indent=2,
    )

print("=== Aggregate faithfulness metrics ===")

for method, metrics in faithfulness_aggregate.items():
    print(f"\n{method.upper()}")
    for key, value in metrics.items():
        print(f"{key}: {value}")

print("Aggregate faithfulness metrics saved.")
print("Path:", FAITHFULNESS_AGGREGATE_PATH)
print("File exists:", FAITHFULNESS_AGGREGATE_PATH.exists())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# Stability

## Read xai records

In [15]:
import json
from pathlib import Path
from busi_xai.config import XAI_OUTPUT_DIR

XAI_RECORDS_PATH = XAI_OUTPUT_DIR / "xai_records.json"

with XAI_RECORDS_PATH.open("r", encoding="utf-8") as f:
    xai_records = json.load(f)

print("=== Existing XAI records ===")
print(f"Total records: {len(xai_records)}")

methods = sorted({record["method"] for record in xai_records})
print(f"Methods: {methods}")

dataset_indices = sorted({
    record["dataset_index"]
    for record in xai_records
})

print(f"Unique dataset indices: {len(dataset_indices)}")
print(f"First 5 indices: {dataset_indices[:5]}")
print(f"Last 5 indices: {dataset_indices[-5:]}")

for method in ["gradcam", "lime", "shap"]:
    method_records = [
        record for record in xai_records
        if record["method"] == method
    ]

    print(
        f"{method.upper()}: {len(method_records)} records"
    )

    first = method_records[0]
    print(
        f"  Example: dataset_index={first['dataset_index']}, "
        f"predicted_class={first['predicted_class']}, "
        f"predicted_probability={first['predicted_probability']}"
    )

=== Existing XAI records ===
Total records: 291
Methods: ['gradcam', 'lime', 'shap']
Unique dataset indices: 97
First 5 indices: [0, 1, 2, 3, 4]
Last 5 indices: [92, 93, 94, 95, 96]
GRADCAM: 97 records
  Example: dataset_index=0, predicted_class=0, predicted_probability=0.990011990070343
LIME: 97 records
  Example: dataset_index=0, predicted_class=0, predicted_probability=0.990011990070343
SHAP: 97 records
  Example: dataset_index=0, predicted_class=0, predicted_probability=0.990011990070343


## Test generate noise

In [17]:
import torch

from busi_xai.xai_evaluation import generate_stability_noise

test_tensor = torch.zeros(
    (1, 3, 224, 224),
    dtype=torch.float32,
)

perturbed_a = generate_stability_noise(
    image_tensor=test_tensor,
    noise_std=0.05,
    seed=42,
)

perturbed_b = generate_stability_noise(
    image_tensor=test_tensor,
    noise_std=0.05,
    seed=42,
)

perturbed_c = generate_stability_noise(
    image_tensor=test_tensor,
    noise_std=0.05,
    seed=43,
)

print("=== Stability noise validation ===")
print(f"Shape: {perturbed_a.shape}")
print(f"Dtype: {perturbed_a.dtype}")
print(f"Mean: {perturbed_a.mean().item():.6f}")
print(f"Std: {perturbed_a.std().item():.6f}")
print(f"Same seed identical: {torch.equal(perturbed_a, perturbed_b)}")
print(f"Different seed different: {not torch.equal(perturbed_a, perturbed_c)}")

=== Stability noise validation ===
Shape: torch.Size([1, 3, 224, 224])
Dtype: torch.float32
Mean: 0.000024
Std: 0.050260
Same seed identical: True
Different seed different: True


## test spearman_correlation

In [ ]:
import numpy as np

from busi_xai.xai_evaluation import (
    calculate_map_spearman_correlation,
    calculate_map_mae,
)

original_map = np.array(
    [[0.0, 0.2],
     [0.5, 1.0]],
    dtype=np.float32,
)

identical_map = original_map.copy()

different_map = np.array(
    [[1.0, 0.5],
     [0.2, 0.0]],
    dtype=np.float32,
)

print("=== Stability metric validation ===")

identical_spearman = calculate_map_spearman_correlation(
    original_map,
    identical_map,
)

identical_mae = calculate_map_mae(
    original_map,
    identical_map,
)

different_spearman = calculate_map_spearman_correlation(
    original_map,
    different_map,
)

different_mae = calculate_map_mae(
    original_map,
    different_map,
)

print(f"Identical maps - Spearman: {identical_spearman}")
print(f"Identical maps - MAE: {identical_mae}")
print(f"Different maps - Spearman: {different_spearman}")
print(f"Different maps - MAE: {different_mae}")

=== Stability metric validation ===
Identical maps - Spearman: 1.0
Identical maps - MAE: 0.0
Different maps - Spearman: -1.0
Different maps - MAE: 0.6499999761581421


## Load test dataset

In [ ]:
import pandas as pd

from busi_xai.data import BUSIDataset, build_image_transform
from busi_xai.config import SPLITS_DIR

test_df = pd.read_csv(SPLITS_DIR / "test.csv")

test_dataset = BUSIDataset(
    dataframe=test_df,
    image_column="processed_image",
    class_column="class_name",
    image_transform=build_image_transform(train=False),
    mask_column="processed_mask",
)

print("=== Stability test dataset ===")
print(f"Test dataframe rows: {len(test_df)}")
print(f"Dataset length: {len(test_dataset)}")

sample_index = 0
sample = test_dataset[sample_index]

print(f"Image shape: {sample['image'].shape}")
print(f"Image dtype: {sample['image'].dtype}")
print(f"Label: {sample['label']}")
print(f"Dataset index: {sample['index']}")
print(f"Mask shape: {sample['mask'].shape}")

=== Stability test dataset ===
Test dataframe rows: 97
Dataset length: 97
Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: 0
Dataset index: 0
Mask shape: torch.Size([1, 224, 224])


## Load model

In [ ]:
import torch
from busi_xai.config import MODELS_DIR
from busi_xai.model import build_resnet50, load_checkpoint


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_resnet50(num_classes=2)

checkpoint_path = MODELS_DIR / "resnet50_best.pt"

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

model = model.to(device)

model.eval()

print("=== Stability model ===")
print(f"Device: {device}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Training mode: {model.training}")

parameters_require_grad = any(
    parameter.requires_grad
    for parameter in model.parameters()
)

print(f"Any parameters require gradients: {parameters_require_grad}")

=== Stability model ===
Device: cuda
Checkpoint: /content/drive/MyDrive/BUSI-XAI/outputs/models/resnet50_best.pt
Training mode: False
Any parameters require gradients: True


## perturb sample image

In [ ]:
import torch

from busi_xai.config import XAI_STABILITY_NOISE_STD, XAI_STABILITY_RANDOM_SEED
from busi_xai.xai import generate_stability_noise

image = sample["image"].unsqueeze(0).to(device)

perturbed_image = generate_stability_noise(
    image_tensor=image,
    noise_std=XAI_STABILITY_NOISE_STD,
    seed=XAI_STABILITY_RANDOM_SEED,
)

noise = perturbed_image - image

print("=== First stability perturbation ===")
print(f"Original shape: {image.shape}")
print(f"Perturbed shape: {perturbed_image.shape}")
print(f"Noise mean: {noise.mean().item():.6f}")
print(f"Noise std: {noise.std().item():.6f}")
print(f"Original unchanged: {torch.equal(image, sample['image'].unsqueeze(0).to(device))}")
print(f"Perturbed differs from original: {not torch.equal(image, perturbed_image)}")

=== First stability perturbation ===
Original shape: torch.Size([1, 3, 224, 224])
Perturbed shape: torch.Size([1, 3, 224, 224])
Noise mean: -0.000202
Noise std: 0.049892
Original unchanged: True
Perturbed differs from original: True


## calculate probabilites after perturbation

In [ ]:
with torch.no_grad():
    original_logits = model(image)
    perturbed_logits = model(perturbed_image)

original_probabilities = torch.softmax(
    original_logits,
    dim=1,
)

perturbed_probabilities = torch.softmax(
    perturbed_logits,
    dim=1,
)

original_predicted_class = int(
    torch.argmax(original_probabilities, dim=1).item()
)

perturbed_predicted_class = int(
    torch.argmax(perturbed_probabilities, dim=1).item()
)

original_probability = float(
    original_probabilities[0, original_predicted_class].item()
)

perturbed_probability = float(
    perturbed_probabilities[0, original_predicted_class].item()
)

print("=== Stability prediction check ===")
print(f"Original predicted class: {original_predicted_class}")
print(f"Original predicted probability: {original_probability:.6f}")
print(f"Perturbed predicted class: {perturbed_predicted_class}")
print(
    f"Perturbed probability for original target class: "
    f"{perturbed_probability:.6f}"
)
print(
    f"Prediction changed: "
    f"{original_predicted_class != perturbed_predicted_class}"
)

=== Stability prediction check ===
Original predicted class: 0
Original predicted probability: 0.990012
Perturbed predicted class: 0
Perturbed probability for original target class: 0.995192
Prediction changed: False


In [ ]:
import numpy as np

from busi_xai.xai import (
    generate_gradcam,
    generate_lime,
    generate_shap,
)

target_class = original_predicted_class

print("=== Perturbed explanation generation ===")

perturbed_gradcam, runtime_seconds = generate_gradcam(
    model=model,
    image=perturbed_image,
    target_class=target_class,
    device=device
)

perturbed_lime, runtime_seconds = generate_lime(
    model=model,
    image_tensor=perturbed_image,
    device=device,
    target_class=target_class,
    random_seed=42,
)

perturbed_shap, runtime_seconds = generate_shap(
    model=model,
    image_tensor=perturbed_image,
    device=device,
    target_class=target_class,
    random_seed=42,
)

perturbed_maps = {
    "gradcam": perturbed_gradcam,
    "lime": perturbed_lime,
    "shap": perturbed_shap,
}

for method, attribution_map in perturbed_maps.items():
    attribution_map = np.asarray(attribution_map)

    print(f"\n{method.upper()}")
    print(f"Shape: {attribution_map.shape}")
    print(f"Dtype: {attribution_map.dtype}")
    print(f"Min: {attribution_map.min()}")
    print(f"Max: {attribution_map.max()}")

=== Perturbed explanation generation ===


  0%|          | 0/1000 [00:00<?, ?it/s]


GRADCAM
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0

LIME
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0

SHAP
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0


In [ ]:
from pathlib import Path
from busi_xai.config import XAI_OUTPUT_DIR

import numpy as np

from busi_xai.xai import (
    calculate_map_spearman_correlation,
    calculate_map_mae,
)

dataset_index = 0

print("=== Sample 0 stability metrics ===")

for method in ["gradcam", "lime", "shap"]:

    original_map_path = (
        XAI_OUTPUT_DIR
        / method
        / f"dataset_index_{dataset_index:03d}.npy"
    )

    original_map = np.load(original_map_path)
    perturbed_map = perturbed_maps[method]

    spearman = calculate_map_spearman_correlation(
        original_map,
        perturbed_map,
    )

    mae = calculate_map_mae(
        original_map,
        perturbed_map,
    )

    print(f"\n{method.upper()}")
    print(f"Spearman correlation: {spearman:.6f}")
    print(f"MAE: {mae:.6f}")

=== Sample 0 stability metrics ===

GRADCAM
Spearman correlation: 0.665421
MAE: 0.038872

LIME
Spearman correlation: 0.382609
MAE: 0.168370

SHAP
Spearman correlation: 0.576487
MAE: 0.016229


In [ ]:
import numpy as np

print("=== Sample 0 stability validation ===")

for method in ["gradcam", "lime", "shap"]:
    attribution_map = np.asarray(perturbed_maps[method])

    spearman = calculate_map_spearman_correlation(
        np.load(
            XAI_OUTPUT_DIR
            / method
            / "dataset_index_000.npy"
        ),
        attribution_map,
    )

    mae = calculate_map_mae(
        np.load(
            XAI_OUTPUT_DIR
            / method
            / "dataset_index_000.npy"
        ),
        attribution_map,
    )

    print(f"\n{method.upper()}")
    print(f"Map finite: {np.isfinite(attribution_map).all()}")
    print(f"Map range valid: {0.0 <= attribution_map.min() <= 1.0 and 0.0 <= attribution_map.max() <= 1.0}")
    print(f"Spearman finite: {np.isfinite(spearman)}")
    print(f"MAE finite: {np.isfinite(mae)}")

=== Sample 0 stability validation ===

GRADCAM
Map finite: True
Map range valid: True
Spearman finite: True
MAE finite: True

LIME
Map finite: True
Map range valid: True
Spearman finite: True
MAE finite: True

SHAP
Map finite: True
Map range valid: True
Spearman finite: True
MAE finite: True


In [ ]:
from busi_xai.xai import validate_xai_stability_records

test_records = []

for method in ["gradcam", "lime", "shap"]:
    test_records.append({
        "image_id": 0,
        "dataset_index": 0,
        "true_label": 0,
        "original_predicted_class": 0,
        "perturbed_predicted_class": 0,
        "prediction_changed": False,
        "original_probability": 0.99,
        "perturbed_probability_for_original_target": 0.98,
        "method": method,
        "spearman_correlation": 0.7,
        "mae": 0.05,
        "noise_std": 0.05,
        "random_seed": 42,
    })

validation = validate_xai_stability_records(
    records=test_records,
    expected_num_samples=1,
)

print("=== Stability record validator test ===")
print(f"Valid: {validation['valid']}")
print(f"Record count: {validation['record_count']}")
print(f"Records per method: {validation['records_per_method']}")
print(f"Errors: {validation['errors']}")

=== Stability record validator test ===
Valid: True
Record count: 3
Records per method: {'gradcam': 1, 'lime': 1, 'shap': 1}
Errors: []


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch

from busi_xai.config import (
    XAI_STABILITY_NOISE_STD,
    XAI_STABILITY_RANDOM_SEED,
)
from busi_xai.xai import (
    generate_gradcam,
    generate_lime,
    generate_shap,
    build_xai_stability_record,
    calculate_map_spearman_correlation,
    calculate_map_mae,
    validate_xai_stability_records,
    generate_stability_noise
)

# Existing records and maps
XAI_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/BUSI-XAI/outputs/xai"
)

with (
    XAI_OUTPUT_DIR / "xai_records.json"
).open("r", encoding="utf-8") as f:
    existing_records = json.load(f)

existing_by_key = {
    (record["dataset_index"], record["method"]): record
    for record in existing_records
}

# Use sample 0 only for this pipeline test.
dataset_index = 0
sample = test_dataset[dataset_index]

image = sample["image"].unsqueeze(0).to(device)

base_record = next(
    record
    for record in existing_records
    if record["dataset_index"] == dataset_index
    and record["method"] == "gradcam"
)

original_predicted_class = int(
    base_record["predicted_class"]
)

original_probability = float(
    base_record["predicted_probability"]
)

true_label = int(sample["label"])

# Same perturbation reused by all three methods.
perturbed_image = generate_stability_noise(
    image_tensor=image,
    noise_std=XAI_STABILITY_NOISE_STD,
    seed=XAI_STABILITY_RANDOM_SEED + dataset_index,
)

with torch.no_grad():
    perturbed_logits = model(perturbed_image)
    perturbed_probabilities = torch.softmax(
        perturbed_logits,
        dim=1,
    )

perturbed_predicted_class = int(
    torch.argmax(
        perturbed_probabilities,
        dim=1,
    ).item()
)

perturbed_probability = float(
    perturbed_probabilities[
        0,
        original_predicted_class,
    ].item()
)

records = []

for method in ["gradcam", "lime", "shap"]:

    start_time = time.perf_counter()

    if method == "gradcam":
        perturbed_map, runtime_seconds = generate_gradcam(
            model=model,
            image=perturbed_image,
            target_class=original_predicted_class,
            device=device,
        )

    elif method == "lime":
        perturbed_map, runtime_seconds = generate_lime(
            model=model,
            image_tensor=perturbed_image,
            device=device,
            target_class=original_predicted_class,
            random_seed=XAI_STABILITY_RANDOM_SEED,
        )

    elif method == "shap":
        perturbed_map, runtime_seconds = generate_shap(
            model=model,
            image_tensor=perturbed_image,
            device=device,
            target_class=original_predicted_class,
            random_seed=XAI_STABILITY_RANDOM_SEED,
        )

    runtime_seconds = time.perf_counter() - start_time

    original_map = np.load(
        XAI_OUTPUT_DIR
        / method
        / f"dataset_index_{dataset_index:03d}.npy"
    )

    perturbed_map = np.asarray(
        perturbed_map,
        dtype=np.float32,
    )

    spearman = calculate_map_spearman_correlation(
        original_map,
        perturbed_map,
    )

    mae = calculate_map_mae(
        original_map,
        perturbed_map,
    )

    record = build_xai_stability_record(
        image_id=dataset_index,
        dataset_index=dataset_index,
        true_label=true_label,
        original_predicted_class=original_predicted_class,
        perturbed_predicted_class=perturbed_predicted_class,
        original_probability=original_probability,
        perturbed_probability=perturbed_probability,
        method=method,
        spearman_correlation=spearman,
        mae=mae,
        noise_std=XAI_STABILITY_NOISE_STD,
        random_seed=XAI_STABILITY_RANDOM_SEED + dataset_index,
    )

    record["runtime_seconds"] = float(runtime_seconds)

    records.append(record)

validation = validate_xai_stability_records(
    records=records,
    expected_num_samples=1,
)

print("=== Single-sample stability pipeline test ===")
print(f"Records generated: {len(records)}")
print(f"Validation valid: {validation['valid']}")
print(f"Validation errors: {validation['errors']}")

for record in records:
    print(
        f"{record['method'].upper()}: "
        f"Spearman={record['spearman_correlation']:.6f}, "
        f"MAE={record['mae']:.6f}, "
        f"runtime={record['runtime_seconds']:.3f}s"
    )

  0%|          | 0/1000 [00:00<?, ?it/s]

=== Single-sample stability pipeline test ===
Records generated: 3
Validation valid: True
Validation errors: []
GRADCAM: Spearman=0.665421, MAE=0.038872, runtime=0.024s
LIME: Spearman=0.382609, MAE=0.168370, runtime=9.282s
SHAP: Spearman=0.576487, MAE=0.016229, runtime=1.902s


In [ ]:
import torch
import pandas as pd

from busi_xai.config import SPLITS_DIR, MODELS_DIR
from busi_xai.data import BUSIDataset, build_image_transform
from busi_xai.model import build_resnet50, load_checkpoint


# Test dataset
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

test_dataset = BUSIDataset(
    dataframe=test_df,
    image_column="processed_image",
    class_column="class_name",
    image_transform=build_image_transform(train=False),
    mask_column="processed_mask",
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_resnet50(num_classes=2)

checkpoint_path = MODELS_DIR / "resnet50_best.pt"

checkpoint = load_checkpoint(
    model=model,
    checkpoint_path=checkpoint_path,
    device=device,
)

model = model.to(device)

model.eval()

print("=== Stability experiment environment ===")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Model training mode: {model.training}")
print(f"Model device: {next(model.parameters()).device}")

=== Stability experiment environment ===
Device: cuda
GPU: Tesla T4
Test samples: 97
Checkpoint: /content/drive/MyDrive/BUSI-XAI/outputs/models/resnet50_best.pt
Model training mode: False
Model device: cuda:0


In [ ]:
from busi_xai.config import (
    XAI_STABILITY_OUTPUT_DIR,
    STABILITY_GRADCAM_OUTPUT_DIR,
    STABILITY_LIME_OUTPUT_DIR,
    STABILITY_SHAP_OUTPUT_DIR,
)
import numpy as np

from busi_xai.xai import validate_xai_stability_map

print("=== Sample-0 stability map validation ===")

for method, perturbed_map in perturbed_maps.items():
    validation = validate_xai_stability_map(perturbed_map)

    print(f"\n{method.upper()}")
    print(f"Valid: {validation['valid']}")
    print(f"Shape: {validation['shape']}")
    print(f"Dtype: {validation['dtype']}")
    print(f"Min: {validation['min']}")
    print(f"Max: {validation['max']}")
    print(f"Errors: {validation['errors']}")

=== Sample-0 stability map validation ===

GRADCAM
Valid: True
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0
Errors: []

LIME
Valid: True
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0
Errors: []

SHAP
Valid: True
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0
Errors: []


In [ ]:
import json
from pathlib import Path

from busi_xai.config import (
    XAI_OUTPUT_DIR,
    GRADCAM_OUTPUT_DIR,
    LIME_OUTPUT_DIR,
    SHAP_OUTPUT_DIR,
)

xai_records_path = XAI_OUTPUT_DIR / "xai_records.json"

with open(xai_records_path, "r") as f:
    xai_records = json.load(f)

method_map_dirs = {
    "gradcam": GRADCAM_OUTPUT_DIR,
    "lime": LIME_OUTPUT_DIR,
    "shap": SHAP_OUTPUT_DIR,
}

print("=== Original XAI artifacts ===")
print(f"Records: {len(xai_records)}")

for method, map_dir in method_map_dirs.items():
    map_files = sorted(
        map_dir.glob("dataset_index_*.npy")
    )

    print(
        f"{method.upper()}: "
        f"{len(map_files)} exact dataset-index maps"
    )

print(
    f"Expected records: "
    f"{len(test_dataset) * 3}"
)

=== Original XAI artifacts ===
Records: 291
GRADCAM: 97 exact dataset-index maps
LIME: 97 exact dataset-index maps
SHAP: 97 exact dataset-index maps
Expected records: 291


In [ ]:
import numpy as np

print("=== Original XAI map alignment ===")

alignment_errors = []

for record in xai_records:
    dataset_index = record["dataset_index"]
    method = record["method"]

    map_path = (
        method_map_dirs[method]
        / f"dataset_index_{dataset_index:03d}.npy"
    )

    if not map_path.exists():
        alignment_errors.append(
            f"Missing map: {method}, dataset_index={dataset_index}"
        )
        continue

    attribution_map = np.load(map_path)

    if attribution_map.shape != (224, 224):
        alignment_errors.append(
            f"Invalid shape: {method}, "
            f"dataset_index={dataset_index}, "
            f"shape={attribution_map.shape}"
        )

    if attribution_map.dtype != np.float32:
        alignment_errors.append(
            f"Invalid dtype: {method}, "
            f"dataset_index={dataset_index}, "
            f"dtype={attribution_map.dtype}"
        )

    if not np.all(np.isfinite(attribution_map)):
        alignment_errors.append(
            f"Non-finite map: {method}, "
            f"dataset_index={dataset_index}"
        )

    if attribution_map.min() < 0.0 or attribution_map.max() > 1.0:
        alignment_errors.append(
            f"Out-of-range map: {method}, "
            f"dataset_index={dataset_index}"
        )

print(f"Checked records: {len(xai_records)}")
print(f"Alignment errors: {len(alignment_errors)}")

if alignment_errors:
    for error in alignment_errors[:10]:
        print(error)
else:
    print("All original maps are correctly aligned and valid.")

=== Original XAI map alignment ===
Checked records: 291
Alignment errors: 0
All original maps are correctly aligned and valid.


In [ ]:
from busi_xai.xai import run_xai_stability_evaluation

# One-image dataset containing only dataset_index 0
test_df_sample = test_df.iloc[[0]].copy()

test_dataset_sample = BUSIDataset(
    dataframe=test_df_sample,
    image_column="processed_image",
    class_column="class_name",
    image_transform=build_image_transform(train=False),
    mask_column="processed_mask",
)

# Existing original records for dataset_index 0 only
original_records_sample = [
    record
    for record in xai_records
    if record["dataset_index"] == 0
]

output_map_dirs = {
    "gradcam": STABILITY_GRADCAM_OUTPUT_DIR,
    "lime": STABILITY_LIME_OUTPUT_DIR,
    "shap": STABILITY_SHAP_OUTPUT_DIR,
}

print("=== GPU single-sample stability test ===")

sample_stability_records = run_xai_stability_evaluation(
    model=model,
    dataset=test_dataset_sample,
    original_records=original_records_sample,
    original_map_dirs=method_map_dirs,
    output_map_dirs=output_map_dirs,
    device=device,
    noise_std=XAI_STABILITY_NOISE_STD,
    random_seed=XAI_STABILITY_RANDOM_SEED,
)

print("\n=== Single-sample results ===")
print(f"Records: {len(sample_stability_records)}")

for record in sample_stability_records:
    print(
        record["method"].upper(),
        f"| Spearman: {record['spearman_correlation']:.6f}",
        f"| MAE: {record['mae']:.6f}",
        f"| Runtime: {record['runtime_seconds']:.3f}s",
        f"| Prediction changed: {record['prediction_changed']}",
    )

=== GPU single-sample stability test ===


  0%|          | 0/1000 [00:00<?, ?it/s]

Completed dataset_index=0

=== Single-sample results ===
Records: 3
GRADCAM | Spearman: 0.665421 | MAE: 0.038872 | Runtime: 0.024s | Prediction changed: False
LIME | Spearman: 0.382609 | MAE: 0.168370 | Runtime: 9.309s | Prediction changed: False
SHAP | Spearman: 0.576487 | MAE: 0.016229 | Runtime: 1.853s | Prediction changed: False


In [ ]:
from busi_xai.xai import validate_xai_stability_map

print("=== GPU single-sample stability validation ===")

map_errors = []

for method in ("gradcam", "lime", "shap"):
    map_path = (
        output_map_dirs[method]
        / "dataset_index_000.npy"
    )

    attribution_map = np.load(map_path)

    validation = validate_xai_stability_map(
        attribution_map
    )

    print(f"\n{method.upper()}")
    print(f"Valid: {validation['valid']}")
    print(f"Shape: {validation['shape']}")
    print(f"Dtype: {validation['dtype']}")
    print(f"Min: {validation['min']}")
    print(f"Max: {validation['max']}")
    print(f"Errors: {validation['errors']}")

    if not validation["valid"]:
        map_errors.extend(validation["errors"])

record_validation = validate_xai_stability_records(
    sample_stability_records,
    expected_num_samples=1,
    expected_methods=("gradcam", "lime", "shap"),
)

print("\n=== Record validation ===")
print(f"Valid: {record_validation['valid']}")
print(f"Record count: {record_validation['record_count']}")
print(
    f"Records per method: "
    f"{record_validation['records_per_method']}"
)
print(f"Errors: {record_validation['errors']}")

=== GPU single-sample stability validation ===

GRADCAM
Valid: True
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0
Errors: []

LIME
Valid: True
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0
Errors: []

SHAP
Valid: True
Shape: (224, 224)
Dtype: float32
Min: 0.0
Max: 1.0
Errors: []

=== Record validation ===
Valid: True
Record count: 3
Records per method: {'gradcam': 1, 'lime': 1, 'shap': 1}
Errors: []


In [ ]:
print("=== Full XAI stability experiment ===")
print(f"Test samples: {len(test_dataset)}")
print("Methods: gradcam, lime, shap")
print(f"Noise std: {XAI_STABILITY_NOISE_STD}")
print(f"Base random seed: {XAI_STABILITY_RANDOM_SEED}")
print()

stability_records = run_xai_stability_evaluation(
    model=model,
    dataset=test_dataset,
    original_records=xai_records,
    original_map_dirs=method_map_dirs,
    output_map_dirs=output_map_dirs,
    device=device,
    noise_std=XAI_STABILITY_NOISE_STD,
    random_seed=XAI_STABILITY_RANDOM_SEED,
)

print("\n=== Full stability run completed ===")
print(f"Total records: {len(stability_records)}")

=== Full XAI stability experiment ===
Test samples: 97
Methods: gradcam, lime, shap
Noise std: 0.05
Base random seed: 42



  0%|          | 0/1000 [00:00<?, ?it/s]

Completed dataset_index=0


  0%|          | 0/1000 [00:00<?, ?it/s]

Completed dataset_index=1


...

Completed dataset_index=94


  0%|          | 0/1000 [00:00<?, ?it/s]

Completed dataset_index=95


  0%|          | 0/1000 [00:00<?, ?it/s]

Completed dataset_index=96

=== Full stability run completed ===
Total records: 291


In [ ]:
from busi_xai.xai import validate_xai_stability_records

stability_validation = validate_xai_stability_records(
    stability_records,
    expected_num_samples=97,
    expected_methods=("gradcam", "lime", "shap"),
)

print("=== Full stability record validation ===")
print(f"Valid: {stability_validation['valid']}")
print(f"Record count: {stability_validation['record_count']}")
print(
    f"Records per method: "
    f"{stability_validation['records_per_method']}"
)
print(f"Errors: {stability_validation['errors']}")

=== Full stability record validation ===
Valid: True
Record count: 291
Records per method: {'gradcam': 97, 'lime': 97, 'shap': 97}
Errors: []


In [ ]:
print("=== Full stability map validation ===")

map_errors = []
map_counts = {}

for method, map_dir in output_map_dirs.items():
    map_files = sorted(
        map_dir.glob("dataset_index_*.npy")
    )

    map_counts[method] = len(map_files)

    for dataset_index in range(len(test_dataset)):
        map_path = (
            map_dir
            / f"dataset_index_{dataset_index:03d}.npy"
        )

        if not map_path.exists():
            map_errors.append(
                f"Missing map: {method}, "
                f"dataset_index={dataset_index}"
            )
            continue

        attribution_map = np.load(map_path)

        validation = validate_xai_stability_map(
            attribution_map
        )

        if not validation["valid"]:
            map_errors.append(
                f"{method}, dataset_index={dataset_index}: "
                f"{validation['errors']}"
            )

print(f"Map counts: {map_counts}")
print(f"Expected maps per method: {len(test_dataset)}")
print(f"Total map errors: {len(map_errors)}")

if map_errors:
    for error in map_errors[:10]:
        print(error)
else:
    print("All 291 perturbed maps are valid and correctly aligned.")

=== Full stability map validation ===
Map counts: {'gradcam': 97, 'lime': 97, 'shap': 97}
Expected maps per method: 97
Total map errors: 0
All 291 perturbed maps are valid and correctly aligned.


In [ ]:
import json

stability_records_path = (
    XAI_STABILITY_OUTPUT_DIR
    / "xai_stability_records.json"
)

with open(stability_records_path, "w") as f:
    json.dump(
        stability_records,
        f,
        indent=2,
    )

print("=== Stability records saved ===")
print(f"Path: {stability_records_path}")
print(f"Records saved: {len(stability_records)}")

=== Stability records saved ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/xai/stability/xai_stability_records.json
Records saved: 291


In [ ]:
import numpy as np

methods = ("gradcam", "lime", "shap")

stability_aggregate = {}

for method in methods:
    method_records = [
        record
        for record in saved_stability_records
        if record["method"] == method
    ]

    spearman_values = np.array([
        record["spearman_correlation"]
        for record in method_records
    ])

    mae_values = np.array([
        record["mae"]
        for record in method_records
    ])

    runtime_values = np.array([
        record["runtime_seconds"]
        for record in method_records
    ])

    prediction_changed_values = np.array([
        record["prediction_changed"]
        for record in method_records
    ], dtype=np.float32)

    stability_aggregate[method] = {
        "n": len(method_records),
        "spearman_mean": float(np.mean(spearman_values)),
        "spearman_median": float(np.median(spearman_values)),
        "spearman_std": float(np.std(spearman_values)),
        "spearman_min": float(np.min(spearman_values)),
        "spearman_max": float(np.max(spearman_values)),
        "mae_mean": float(np.mean(mae_values)),
        "mae_median": float(np.median(mae_values)),
        "mae_std": float(np.std(mae_values)),
        "mae_min": float(np.min(mae_values)),
        "mae_max": float(np.max(mae_values)),
        "runtime_mean_seconds": float(np.mean(runtime_values)),
        "runtime_median_seconds": float(np.median(runtime_values)),
        "prediction_changed_count": int(
            np.sum(prediction_changed_values)
        ),
        "prediction_changed_rate": float(
            np.mean(prediction_changed_values)
        ),
    }

print("=== Aggregate stability metrics ===")

for method, metrics in stability_aggregate.items():
    print(f"\n{method.upper()}")
    for key, value in metrics.items():
        print(f"{key}: {value}")

=== Aggregate stability metrics ===

GRADCAM
n: 97
spearman_mean: 0.6743454702465592
spearman_median: 0.7613215824721162
spearman_std: 0.23599831943482183
spearman_min: -0.06400678792640783
spearman_max: 0.9201818506603627
mae_mean: 0.06465208689807002
mae_median: 0.05677739530801773
mae_std: 0.036663846734408485
mae_min: 0.013887040317058563
mae_max: 0.1658102124929428
runtime_mean_seconds: 0.02789952625773798
runtime_median_seconds: 0.02154515700021875
prediction_changed_count: 17
prediction_changed_rate: 0.17525772750377655

LIME
n: 97
spearman_mean: 0.3779471869427136
spearman_median: 0.38684054231247067
spearman_std: 0.1614356889927064
spearman_min: -0.027023976801663043
spearman_max: 0.7256008256548753
mae_mean: 0.17170370377830624
mae_median: 0.16479799151420593
mae_std: 0.04583766278386432
mae_min: 0.07686237245798111
mae_max: 0.2801475524902344
runtime_mean_seconds: 9.714652198402066
runtime_median_seconds: 9.50754226800018
prediction_changed_count: 17
prediction_changed_rate:

In [ ]:
print("=== Aggregate stability artifact validation ===")

aggregate_errors = []

for method in methods:
    metrics = stability_aggregate[method]

    if metrics["n"] != 97:
        aggregate_errors.append(
            f"{method}: expected n=97, "
            f"found {metrics['n']}"
        )

    for key in (
        "spearman_mean",
        "spearman_median",
        "spearman_std",
        "spearman_min",
        "spearman_max",
    ):
        value = metrics[key]

        if not np.isfinite(value):
            aggregate_errors.append(
                f"{method}: non-finite {key}"
            )

        if value < -1.0 or value > 1.0:
            aggregate_errors.append(
                f"{method}: {key} outside [-1, 1]: {value}"
            )

    for key in (
        "mae_mean",
        "mae_median",
        "mae_std",
        "mae_min",
        "mae_max",
    ):
        value = metrics[key]

        if not np.isfinite(value):
            aggregate_errors.append(
                f"{method}: non-finite {key}"
            )

        if value < 0:
            aggregate_errors.append(
                f"{method}: negative {key}: {value}"
            )

    for key in (
        "runtime_mean_seconds",
        "runtime_median_seconds",
    ):
        value = metrics[key]

        if not np.isfinite(value):
            aggregate_errors.append(
                f"{method}: non-finite {key}"
            )

        if value <= 0:
            aggregate_errors.append(
                f"{method}: invalid {key}: {value}"
            )

    if not (
        0 <= metrics["prediction_changed_count"] <= 97
    ):
        aggregate_errors.append(
            f"{method}: invalid prediction_changed_count"
        )

    if not (
        0.0 <= metrics["prediction_changed_rate"] <= 1.0
    ):
        aggregate_errors.append(
            f"{method}: invalid prediction_changed_rate"
        )

print(f"Methods: {list(stability_aggregate.keys())}")
print(f"Aggregate errors: {len(aggregate_errors)}")

if aggregate_errors:
    for error in aggregate_errors:
        print(error)
else:
    print("Aggregate stability artifact is valid.")

=== Aggregate stability artifact validation ===
Methods: ['gradcam', 'lime', 'shap']
Aggregate errors: 0
Aggregate stability artifact is valid.


In [ ]:
import json
from busi_xai.config import METRICS_DIR
stability_aggregate_path = (
    METRICS_DIR
    / "xai_stability_aggregate_metrics.json"
)

with open(stability_aggregate_path, "w") as f:
    json.dump(
        stability_aggregate,
        f,
        indent=2,
    )

print("=== Aggregate stability metrics saved ===")
print(f"Path: {stability_aggregate_path}")
print(f"Methods saved: {list(stability_aggregate.keys())}")

=== Aggregate stability metrics saved ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_stability_aggregate_metrics.json
Methods saved: ['gradcam', 'lime', 'shap']


In [ ]:
import json

with open(stability_aggregate_path, "r") as f:
    saved_stability_aggregate = json.load(f)

print("=== Saved aggregate stability validation ===")
print(
    "Matches in-memory results:",
    saved_stability_aggregate == stability_aggregate,
)
print(
    "Methods:",
    list(saved_stability_aggregate.keys()),
)

saved_aggregate_errors = []

for method in methods:
    if method not in saved_stability_aggregate:
        saved_aggregate_errors.append(
            f"Missing method: {method}"
        )
        continue

    if saved_stability_aggregate[method]["n"] != 97:
        saved_aggregate_errors.append(
            f"{method}: invalid n"
        )

print(f"Validation errors: {len(saved_aggregate_errors)}")

if saved_aggregate_errors:
    for error in saved_aggregate_errors:
        print(error)
else:
    print("Saved aggregate artifact is valid.")

=== Saved aggregate stability validation ===
Matches in-memory results: True
Methods: ['gradcam', 'lime', 'shap']
Validation errors: 0
Saved aggregate artifact is valid.


# Pairwise explanation agreement

In [ ]:
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/BUSI-XAI")

XAI_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "xai"

METHODS = ["gradcam", "lime", "shap"]
N_TEST = 97

map_dirs = {
    method: XAI_OUTPUT_DIR / method
    for method in METHODS
}


def load_original_xai_map(method, dataset_index):
    path = map_dirs[method] / f"dataset_index_{dataset_index:03d}.npy"
    return np.load(path)


def calculate_pairwise_spearman(map_a, map_b):
    a = np.asarray(map_a, dtype=np.float64).ravel()
    b = np.asarray(map_b, dtype=np.float64).ravel()

    if a.shape != b.shape:
        raise ValueError(
            f"Map shape mismatch: {a.shape} vs {b.shape}"
        )

    # Rank transformation using scipy, consistent with the
    # Spearman metric already used in the stability analysis.
    from scipy.stats import spearmanr

    result = spearmanr(a, b)

    if not np.isfinite(result.statistic):
        raise ValueError("Non-finite Spearman correlation.")

    return float(result.statistic)


pairwise_results = []

method_pairs = [
    ("gradcam", "lime"),
    ("gradcam", "shap"),
    ("lime", "shap"),
]

for dataset_index in range(N_TEST):

    maps = {
        method: load_original_xai_map(method, dataset_index)
        for method in METHODS
    }

    for method_a, method_b in method_pairs:
        correlation = calculate_pairwise_spearman(
            maps[method_a],
            maps[method_b],
        )

        pairwise_results.append({
            "dataset_index": dataset_index,
            "method_a": method_a,
            "method_b": method_b,
            "spearman": correlation,
        })


print("=== Pairwise explanation agreement ===")
print(f"Records: {len(pairwise_results)}")

for method_a, method_b in method_pairs:
    values = np.array([
        r["spearman"]
        for r in pairwise_results
        if r["method_a"] == method_a
        and r["method_b"] == method_b
    ])

    print(
        f"{method_a.upper()} vs {method_b.upper()}: "
        f"n={len(values)}, "
        f"mean={values.mean():.6f}, "
        f"median={np.median(values):.6f}, "
        f"std={values.std():.6f}, "
        f"min={values.min():.6f}, "
        f"max={values.max():.6f}"
    )

=== Pairwise explanation agreement ===
Records: 291
GRADCAM vs LIME: n=97, mean=0.231127, median=0.266474, std=0.205670, min=-0.282526, max=0.605498
GRADCAM vs SHAP: n=97, mean=0.340051, median=0.362514, std=0.160252, min=-0.176533, max=0.679113
LIME vs SHAP: n=97, mean=0.162949, median=0.160947, std=0.125945, min=-0.243136, max=0.386939


In [ ]:
import numpy as np

from busi_xai.xai import (
    calculate_pairwise_explanation_spearman,
    build_pairwise_explanation_agreement_record,
    validate_pairwise_explanation_agreement_records,
)

METHODS = ["gradcam", "lime", "shap"]

METHOD_PAIRS = [
    ("gradcam", "lime"),
    ("gradcam", "shap"),
    ("lime", "shap"),
]

EXPECTED_DATASET_INDICES = list(range(97))

pairwise_agreement_records = []

for dataset_index in EXPECTED_DATASET_INDICES:

    maps = {
        method: load_original_xai_map(method, dataset_index)
        for method in METHODS
    }

    for method_a, method_b in METHOD_PAIRS:

        spearman = calculate_pairwise_explanation_spearman(
            maps[method_a],
            maps[method_b],
        )

        record = build_pairwise_explanation_agreement_record(
            dataset_index=dataset_index,
            method_a=method_a,
            method_b=method_b,
            spearman=spearman,
        )

        pairwise_agreement_records.append(record)


validation_errors = validate_pairwise_explanation_agreement_records(
    records=pairwise_agreement_records,
    expected_dataset_indices=EXPECTED_DATASET_INDICES,
    expected_method_pairs=METHOD_PAIRS,
)

print("=== Pairwise explanation agreement validation ===")
print(f"Total records: {len(pairwise_agreement_records)}")
print(f"Expected records: {97 * 3}")
print(f"Validation errors: {len(validation_errors)}")

if validation_errors:
    for error in validation_errors[:20]:
        print(error)
else:
    print("All pairwise agreement records are valid.")

=== Pairwise explanation agreement validation ===
Total records: 291
Expected records: 291
Validation errors: 0
All pairwise agreement records are valid.


In [ ]:
import json
from pathlib import Path
import numpy as np

PAIRWISE_RECORDS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "xai"
    / "xai_pairwise_agreement_records.json"
)

PAIRWISE_AGGREGATE_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "xai_pairwise_agreement_aggregate_metrics.json"
)


# ---------------------------------------------------------
# Save per-image pairwise records
# ---------------------------------------------------------

with open(PAIRWISE_RECORDS_PATH, "w") as f:
    json.dump(pairwise_agreement_records, f, indent=2)


# ---------------------------------------------------------
# Build aggregate metrics
# ---------------------------------------------------------

pairwise_aggregate = {}

for method_a, method_b in METHOD_PAIRS:

    values = np.array([
        record["spearman"]
        for record in pairwise_agreement_records
        if record["method_a"] == method_a
        and record["method_b"] == method_b
    ], dtype=np.float64)

    pair_key = f"{method_a}_vs_{method_b}"

    pairwise_aggregate[pair_key] = {
        "n": int(len(values)),
        "spearman_mean": float(values.mean()),
        "spearman_median": float(np.median(values)),
        "spearman_std": float(values.std()),
        "spearman_min": float(values.min()),
        "spearman_max": float(values.max()),
    }


with open(PAIRWISE_AGGREGATE_PATH, "w") as f:
    json.dump(pairwise_aggregate, f, indent=2)


print("=== Pairwise agreement artifacts saved ===")
print(f"Records: {PAIRWISE_RECORDS_PATH}")
print(f"Aggregate: {PAIRWISE_AGGREGATE_PATH}")
print(f"Records saved: {len(pairwise_agreement_records)}")
print(f"Pairs saved: {len(pairwise_aggregate)}")

=== Pairwise agreement artifacts saved ===
Records: /content/drive/MyDrive/BUSI-XAI/outputs/xai/xai_pairwise_agreement_records.json
Aggregate: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_pairwise_agreement_aggregate_metrics.json
Records saved: 291
Pairs saved: 3


# Computational efficiency

In [ ]:
from busi_xai.xai import calculate_xai_runtime_summary
import json
import numpy as np
from busi_xai.config import XAI_DIR, METRICS_DIR

XAI_RECORDS_PATH = (
    XAI_DIR
    / "xai_records.json"
)

with open(XAI_RECORDS_PATH, "r") as f:
    original_xai_records = json.load(f)

methods = ("gradcam", "lime", "shap")
runtime_summary = calculate_xai_runtime_summary(
    records=original_xai_records,
    methods=methods,
)


for method in methods:
    result = runtime_summary[method]

    print(
        f"{method.upper()}: "
        f"n={result['n']}, "
        f"mean={result['runtime_mean_seconds']:.6f}s, "
        f"median={result['runtime_median_seconds']:.6f}s, "
        f"std={result['runtime_std_seconds']:.6f}s, "
        f"min={result['runtime_min_seconds']:.6f}s, "
        f"max={result['runtime_max_seconds']:.6f}s"
    )

expected_n = 97

XAI_RUNTIME_AGGREGATE_PATH = (
    METRICS_DIR
    / "xai_runtime_aggregate_metrics.json"
)

with open(XAI_RUNTIME_AGGREGATE_PATH, "w") as f:
    json.dump(runtime_summary, f, indent=2)

print("=== XAI runtime artifact saved ===")
print(f"Path: {XAI_RUNTIME_AGGREGATE_PATH}")
print(f"Methods saved: {list(runtime_summary.keys())}")

GRADCAM: n=97, mean=0.025664s, median=0.023835s, std=0.005282s, min=0.019676s, max=0.046867s
LIME: n=97, mean=9.827661s, median=9.774137s, std=0.508945s, min=8.963715s, max=11.585494s
SHAP: n=97, mean=2.039720s, median=2.030152s, std=0.053575s, min=1.892659s, max=2.181165s
=== XAI runtime artifact saved ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_runtime_aggregate_metrics.json
Methods saved: ['gradcam', 'lime', 'shap']


# Classification–explanation relationship

In [ ]:
import json
from busi_xai.config import XAI_OUTPUT_DIR, XAI_METHODS
import pandas as pd

XAI_RECORDS_PATH = (
    XAI_OUTPUT_DIR
    / "xai_records.json"
)
FAITHFULNESS_RECORDS_PATH = (
    XAI_OUTPUT_DIR
    / "xai_faithfulness_records.json"
)

STABILITY_RECORDS_PATH = (
    XAI_OUTPUT_DIR
    / "stability"
    / "xai_stability_records.json"
)

with open(XAI_RECORDS_PATH, "r") as f:
    original_records = json.load(f)

with open(FAITHFULNESS_RECORDS_PATH, "r") as f:
    faithfulness_records = json.load(f)

with open(STABILITY_RECORDS_PATH, "r") as f:
    stability_records = json.load(f)

def record_keys(records):
    return {
        (
            int(record["dataset_index"]),
            str(record["method"]),
        )
        for record in records
    }


original_keys = record_keys(original_records)
faithfulness_keys = record_keys(faithfulness_records)
stability_keys = record_keys(stability_records)

# ---------------------------------------------------------
# Index existing records
# ---------------------------------------------------------

original_lookup = {
    (int(record["dataset_index"]), str(record["method"])): record
    for record in original_records
}

faithfulness_lookup = {
    (int(record["dataset_index"]), str(record["method"])): record
    for record in faithfulness_records
}

stability_lookup = {
    (int(record["dataset_index"]), str(record["method"])): record
    for record in stability_records
}


# ---------------------------------------------------------
# Build joined RQ3 dataset
# ---------------------------------------------------------

rq3_rows = []

for key in sorted(original_lookup.keys()):

    original = original_lookup[key]
    faithfulness = faithfulness_lookup[key]
    stability = stability_lookup[key]

    rq3_rows.append({
        "dataset_index": key[0],
        "method": key[1],

        "true_label": int(original["true_label"]),
        "predicted_class": int(original["predicted_class"]),
        "correct": bool(original["correct"]),
        "predicted_probability": float(
            original["predicted_probability"]
        ),

        "iou": float(original["iou"]),
        "dice": float(original["dice"]),

        "probability_drop": float(
            faithfulness["probability_drop"]
        ),

        "stability_spearman": float(
            stability["spearman_correlation"]
        ),
    })


rq3_df = pd.DataFrame(rq3_rows)


# ---------------------------------------------------------
# Validate joined dataset
# ---------------------------------------------------------

expected_rows = 97 * 3

print("=== RQ3 joined dataset validation ===")
print(f"Rows: {len(rq3_df)}")
print(f"Expected rows: {expected_rows}")
print(f"Methods: {sorted(rq3_df['method'].unique().tolist())}")
print(f"Unique dataset indices: {rq3_df['dataset_index'].nunique()}")
print(f"Missing values: {int(rq3_df.isna().sum().sum())}")

expected_columns = {
    "dataset_index",
    "method",
    "true_label",
    "predicted_class",
    "correct",
    "predicted_probability",
    "iou",
    "dice",
    "probability_drop",
    "stability_spearman",
}

print(
    "Columns valid:",
    set(rq3_df.columns) == expected_columns,
)

assert len(rq3_df) == expected_rows
assert rq3_df["dataset_index"].nunique() == 97
assert set(rq3_df["method"]) == set(XAI_METHODS)
assert rq3_df.isna().sum().sum() == 0
assert set(rq3_df.columns) == expected_columns

print("RQ3 joined dataset is valid.")

=== RQ3 joined dataset validation ===
Rows: 291
Expected rows: 291
Methods: ['gradcam', 'lime', 'shap']
Unique dataset indices: 97
Missing values: 0
Columns valid: True
RQ3 joined dataset is valid.


In [ ]:
# ---------------------------------------------------------
# RQ3 — Explanation quality by classification correctness
# ---------------------------------------------------------

quality_metrics = [
    "iou",
    "dice",
    "probability_drop",
    "stability_spearman",
]

print("=== RQ3 correctness-group summary ===")

rq3_correctness_summary = (
    rq3_df
    .groupby(["method", "correct"])[quality_metrics]
    .agg(["count", "mean", "median", "std"])
)

print(rq3_correctness_summary)

# ---------------------------------------------------------
# Validate correctness groups
# ---------------------------------------------------------

print("\n=== RQ3 correctness-group validation ===")

print(
    "Correct classifications:",
    int(rq3_df["correct"].sum())
)

print(
    "Incorrect classifications:",
    int((~rq3_df["correct"]).sum())
)

print(
    "Correctness values:",
    sorted(rq3_df["correct"].unique().tolist())
)

assert set(rq3_df["correct"].unique()) == {True, False}
assert rq3_df.groupby(["method", "correct"]).size().shape[0] == 6

print("RQ3 correctness-group analysis is valid.")

=== RQ3 correctness-group summary ===
                  iou                                dice                      \
                count      mean    median       std count      mean    median   
method  correct                                                                 
gradcam False      10  0.200444  0.202589  0.158348    10  0.307954  0.336799   
        True       87  0.189039  0.164619  0.158962    87  0.289242  0.282700   
lime    False      10  0.141275  0.145203  0.127542    10  0.228020  0.253517   
        True       87  0.138069  0.075361  0.165229    87  0.211561  0.140159   
shap    False      10  0.158868  0.132201  0.109010    10  0.260936  0.233139   
        True       87  0.178188  0.157272  0.125684    87  0.284060  0.271797   

                          probability_drop                                \
                      std            count      mean    median       std   
method  correct                                                            
grad

In [ ]:
from scipy.stats import spearmanr

# ---------------------------------------------------------
# RQ3 — Predicted probability vs explanation quality
# ---------------------------------------------------------

correlation_metrics = [
    "iou",
    "dice",
    "probability_drop",
    "stability_spearman",
]

rq3_probability_correlations = []

for method in XAI_METHODS:

    method_df = rq3_df[rq3_df["method"] == method]

    for metric in correlation_metrics:

        rho, p_value = spearmanr(
            method_df["predicted_probability"],
            method_df[metric],
        )

        rq3_probability_correlations.append({
            "method": method,
            "metric": metric,
            "spearman_rho": float(rho),
            "p_value": float(p_value),
            "n": int(len(method_df)),
        })


rq3_probability_correlations_df = pd.DataFrame(
    rq3_probability_correlations
)

print("=== RQ3 probability–explanation correlations ===")
print(
    rq3_probability_correlations_df.to_string(index=False)
)

# ---------------------------------------------------------
# Validate correlation analysis
# ---------------------------------------------------------

expected_correlation_rows = len(XAI_METHODS) * len(
    correlation_metrics
)

print("\n=== RQ3 correlation validation ===")
print(
    "Rows:",
    len(rq3_probability_correlations_df)
)
print(
    "Expected rows:",
    expected_correlation_rows
)
print(
    "Methods:",
    sorted(
        rq3_probability_correlations_df["method"].unique().tolist()
    )
)
print(
    "Metrics:",
    sorted(
        rq3_probability_correlations_df["metric"].unique().tolist()
    )
)
print(
    "Missing values:",
    int(
        rq3_probability_correlations_df.isna().sum().sum()
    )
)

assert (
    len(rq3_probability_correlations_df)
    == expected_correlation_rows
)
assert set(
    rq3_probability_correlations_df["method"]
) == set(XAI_METHODS)
assert set(
    rq3_probability_correlations_df["metric"]
) == set(correlation_metrics)
assert (
    rq3_probability_correlations_df.isna().sum().sum()
    == 0
)

print("RQ3 correlation analysis is valid.")

=== RQ3 probability–explanation correlations ===
 method             metric  spearman_rho      p_value  n
gradcam                iou      0.072720 4.790324e-01 97
gradcam               dice      0.072720 4.790324e-01 97
gradcam   probability_drop     -0.294275 3.436008e-03 97
gradcam stability_spearman      0.707316 5.582055e-16 97
   lime                iou      0.235773 2.007886e-02 97
   lime               dice      0.235773 2.007886e-02 97
   lime   probability_drop     -0.014767 8.858476e-01 97
   lime stability_spearman      0.059081 5.653991e-01 97
   shap                iou      0.106183 3.005973e-01 97
   shap               dice      0.106183 3.005973e-01 97
   shap   probability_drop      0.332566 8.740807e-04 97
   shap stability_spearman      0.232011 2.221204e-02 97

=== RQ3 correlation validation ===
Rows: 12
Expected rows: 12
Methods: ['gradcam', 'lime', 'shap']
Metrics: ['dice', 'iou', 'probability_drop', 'stability_spearman']
Missing values: 0
RQ3 correlation analysis 

In [ ]:
import json
from pathlib import Path
from busi_xai.config import METRICS_DIR
# ---------------------------------------------------------
# RQ3 — Build complete evidence artifact
# ---------------------------------------------------------

rq3_correctness_summary_dict = {}

for (method, correct), group in rq3_df.groupby(
    ["method", "correct"]
):

    rq3_correctness_summary_dict.setdefault(method, {})[
        str(bool(correct))
    ] = {
        "n": int(len(group)),
        "metrics": {
            metric: {
                "mean": float(group[metric].mean()),
                "median": float(group[metric].median()),
                "std": float(group[metric].std()),
            }
            for metric in quality_metrics
        },
    }


rq3_probability_correlations_dict = (
    rq3_probability_correlations_df
    .to_dict(orient="records")
)


rq3_evidence = {
    "analysis": "classification_explanation_relationship",
    "dataset_size": 97,
    "methods": list(XAI_METHODS),

    "joined_rows": int(len(rq3_df)),

    "classification_counts": {
        "correct": int(rq3_df["correct"].sum()),
        "incorrect": int((~rq3_df["correct"]).sum()),
    },

    "correctness_group_summary": (
        rq3_correctness_summary_dict
    ),

    "probability_correlations": (
        rq3_probability_correlations_dict
    ),
}


# ---------------------------------------------------------
# Save artifact
# ---------------------------------------------------------

rq3_output_path = (
    METRICS_DIR
    / "xai_rq3_classification_explanation_relationship.json"
)

rq3_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(rq3_output_path, "w") as f:
    json.dump(
        rq3_evidence,
        f,
        indent=2,
    )


print("=== RQ3 evidence artifact saved ===")
print(f"Path: {rq3_output_path}")
print(f"Methods: {rq3_evidence['methods']}")
print(f"Joined rows: {rq3_evidence['joined_rows']}")

=== RQ3 evidence artifact saved ===
Path: /content/drive/MyDrive/BUSI-XAI/outputs/metrics/xai_rq3_classification_explanation_relationship.json
Methods: ['gradcam', 'lime', 'shap']
Joined rows: 291
